In [1]:
from crewai import Agent, Crew, Process, Task
from crewai.project import CrewBase, agent, crew, task
from dotenv import load_dotenv
import os
from crewai import Agent, Task, Crew, Process, LLM
import os
from langchain_openai import ChatOpenAI
from crewai_tools import MCPServerAdapter
from dotenv import load_dotenv
import os
from crewai import LLM
from langchain_openai import ChatOpenAI

/home/ridwanfatur/work/portfolio/modular-ai/venvs/crewai-mcp-server/.venv/lib/python3.12/site-packages/pydantic/fields.py:1093: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'required'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.11/migration/
  warn(


In [2]:
load_dotenv()

True

In [11]:
llm = ChatOpenAI(
    openai_api_base="https://api.groq.com/openai/v1",
    openai_api_key=os.environ.get("GROQ_API_KEY"),
    temperature=0,
    model_name=f"groq/llama-3.3-70b-versatile",
    top_p=1,
    max_retries=3,
    request_timeout=60,
)

agent_backstory = """
You are a data validation assistant.
Your job is to evaluate whether a given input is valid, reliable, and logically consistent.

You carefully inspect:
- Missing or incomplete information
- Contradictions or inconsistencies
- Logical correctness
- Clarity and plausibility

You provide concise explanations and avoid overcomplicating the analysis.
"""

task_description = """
Evaluate the following input data:

{user_input}

Your task:
1. Determine whether the input is valid or not.
2. Give a validity score from 0 to 100.
3. Briefly explain why.
4. Mention any suspicious, inconsistent, or missing parts.

Return the result in this format:

Validity: VALID / INVALID
Score: <0-100>

Reason:
<short explanation>

Issues:
- <issue 1>
- <issue 2>
"""

agent = Agent(
    role="Data Validator",
    goal="Determine whether provided data is valid and estimate how trustworthy it is",
    backstory=agent_backstory,
    max_iter=2,
    verbose=True,
    memory=False,
    llm=llm,
)

task = Task(
    description=task_description,
    expected_output="""
A structured validation result containing:
- VALID or INVALID status
- Validity score (0-100)
- Short explanation
- List of detected issues
""",
    agent=agent,
)

crew = Crew(
    agents=[agent],
    tasks=[task],
    process=Process.sequential,
    verbose=True
)

In [6]:
result = crew.kickoff(
    inputs={
        "user_input": """
        Name: John Doe
        Age: 25
        Email: john@example.com
        Country: Indonesia
        """
    }
)

╭──────────────────────────────────────────── Crew Execution Started ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 517e8933-0ce9-49bd-8d0a-816f4d62c5e0                                                                       │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Data Validator                                                                                          │
│                                                                                                                 │
│  Task:                                                                                                          │
│  Evaluate the following input data:                                                                             │
│                                                                                                                 │
│                                                                                                                 │
│          Name: John Doe                                                                                         │
│          Age: 25                                                                                                │
│          Email: john@example.com                                                                                │
│          Country: Indonesia                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
│  Your task:                                                                                                     │
│  1. Determine whether the input is valid or not.                                                                │
│  2. Give a validity score from 0 to 100.                                                                        │
│  3. Briefly explain why.                                                                                        │
│  4. Mention any suspicious, inconsistent, or missing parts.                                                     │
│                                                                                                                 │
│  Return the result in this format:                                                                              │
│                                                                                                                 │
│  Validity: VALID / INVALID                                                                                      │
│  Score: <0-100>                                                                                                 │
│                                                                                                                 │
│  Reason:                                                                                                        │
│  <short explanation>                                                                                            │
│                                                                                                                 │
│  Issues:                                                                                                        │
│  - <issue 1>                                                                                                    │
│  - <issue 2>                                                                                                    │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Data Validator                                                                                          │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Validity: VALID                                                                                                │
│  Score: 90                                                                                                      │
│                                                                                                                 │
│  Reason:                                                                                                        │
│  The input data appears to be mostly complete and consistent, with a name, age, email, and country provided.    │
│  The email address has a valid format, and the age is within a plausible range. However, the validity score is  │
│  not 100 due to potential issues with data accuracy and completeness.                                           │
│                                                                                                                 │
│  Issues:                                                                                                        │
│  - The email address is not verified, and it is unclear whether it actually belongs to John Doe.                │
│  - There is no additional contact information or verification data to confirm the identity of John Doe.         │
│  - The country of residence is provided, but no address or other location-specific details are included.        │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Task Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: 5b764664-e98c-40a7-8228-13ebef7a8adb                                                                     │
│  Agent: Data Validator                                                                                          │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 517e8933-0ce9-49bd-8d0a-816f4d62c5e0                                                                       │
│  Tool Args:                                                                                                     │
│  Final Output: Validity: VALID                                                                                  │
│  Score: 90                                                                                                      │
│                                                                                                                 │
│  Reason:                                                                                                        │
│  The input data appears to be mostly complete and consistent, with a name, age, email, and country provided.    │
│  The email address has a valid format, and the age is within a plausible range. However, the validity score is  │
│  not 100 due to potential issues with data accuracy and completeness.                                           │
│                                                                                                                 │
│  Issues:                                                                                                        │
│  - The email address is not verified, and it is unclear whether it actually belongs to John Doe.                │
│  - There is no additional contact information or verification data to confirm the identity of John Doe.         │
│  - The country of residence is provided, but no address or other location-specific details are included.        │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [8]:
result.raw

'Validity: VALID\nScore: 90\n\nReason:\nThe input data appears to be mostly complete and consistent, with a name, age, email, and country provided. The email address has a valid format, and the age is within a plausible range. However, the validity score is not 100 due to potential issues with data accuracy and completeness.\n\nIssues:\n- The email address is not verified, and it is unclear whether it actually belongs to John Doe.\n- There is no additional contact information or verification data to confirm the identity of John Doe.\n- The country of residence is provided, but no address or other location-specific details are included.'

In [9]:
result = crew.kickoff(
    inputs={
        "user_input": """
        Name: Unknown
        Age: -5
        Email: not-an-email
        Country: Mars
        Birth Year: 2030
        """
    }
)

╭──────────────────────────────────────────── Crew Execution Started ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 517e8933-0ce9-49bd-8d0a-816f4d62c5e0                                                                       │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Data Validator                                                                                          │
│                                                                                                                 │
│  Task:                                                                                                          │
│  Evaluate the following input data:                                                                             │
│                                                                                                                 │
│                                                                                                                 │
│          Name: Unknown                                                                                          │
│          Age: -5                                                                                                │
│          Email: not-an-email                                                                                    │
│          Country: Mars                                                                                          │
│          Birth Year: 2030                                                                                       │
│                                                                                                                 │
│                                                                                                                 │
│  Your task:                                                                                                     │
│  1. Determine whether the input is valid or not.                                                                │
│  2. Give a validity score from 0 to 100.                                                                        │
│  3. Briefly explain why.                                                                                        │
│  4. Mention any suspicious, inconsistent, or missing parts.                                                     │
│                                                                                                                 │
│  Return the result in this format:                                                                              │
│                                                                                                                 │
│  Validity: VALID / INVALID                                                                                      │
│  Score: <0-100>                                                                                                 │
│                                                                                                                 │
│  Reason:                                                                                                        │
│  <short explanation>                                                                                            │
│                                                                                                                 │
│  Issues:                                                                                                        │
│  - <issue 1>                                                                                                    │
│  - <issue 2>                                                                                                    │
│                                                                                                                 │
│                                                                                                                 │
╰────────────────────────────────────────────────────────

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Data Validator                                                                                          │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Validity: INVALID                                                                                              │
│  Score: 0                                                                                                       │
│                                                                                                                 │
│  Reason:                                                                                                        │
│  The input data contains multiple inconsistencies and invalid values, making it completely unreliable.          │
│                                                                                                                 │
│  Issues:                                                                                                        │
│  - Name is unknown, which is a required field for identification.                                               │
│  - Age is -5, which is not a valid age as it is negative.                                                       │
│  - Email is "not-an-email", which is not a valid email address.                                                 │
│  - Country is Mars, which is not a valid country on Earth.                                                      │
│  - Birth Year is 2030, which is in the future, making it impossible for the person to have been born yet.       │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Task Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: 5b764664-e98c-40a7-8228-13ebef7a8adb                                                                     │
│  Agent: Data Validator                                                                                          │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 517e8933-0ce9-49bd-8d0a-816f4d62c5e0                                                                       │
│  Tool Args:                                                                                                     │
│  Final Output: Validity: INVALID                                                                                │
│  Score: 0                                                                                                       │
│                                                                                                                 │
│  Reason:                                                                                                        │
│  The input data contains multiple inconsistencies and invalid values, making it completely unreliable.          │
│                                                                                                                 │
│  Issues:                                                                                                        │
│  - Name is unknown, which is a required field for identification.                                               │
│  - Age is -5, which is not a valid age as it is negative.                                                       │
│  - Email is "not-an-email", which is not a valid email address.                                                 │
│  - Country is Mars, which is not a valid country on Earth.                                                      │
│  - Birth Year is 2030, which is in the future, making it impossible for the person to have been born yet.       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [10]:
result.raw

'Validity: INVALID\nScore: 0\n\nReason:\nThe input data contains multiple inconsistencies and invalid values, making it completely unreliable.\n\nIssues:\n- Name is unknown, which is a required field for identification.\n- Age is -5, which is not a valid age as it is negative.\n- Email is "not-an-email", which is not a valid email address.\n- Country is Mars, which is not a valid country on Earth.\n- Birth Year is 2030, which is in the future, making it impossible for the person to have been born yet.'